In [1]:
import pathlib
import sys

_root = pathlib.Path().resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
if str(_root.parent) not in sys.path and (_root.parent / 'src').exists():
    sys.path.insert(0, str(_root.parent))


# 04 Geographic Consumer Intelligence

**Research question:** How do consumer purchasing patterns and delivery performance vary across geographic regions?
**Sources:** `analytics.fact_orders` (order grain) joined to `staging.customers` on `customer_id` for order-level `customer_state` (reconciled to all orders). **Filter:** all statuses for volume/value; delivery avg over delivered orders with `delivery_time_days`.

In [2]:
from src.db import get_connection

conn = get_connection(read_only=True)


In [3]:
geo = conn.execute("""
SELECT cu.customer_state AS state,
       COUNT(DISTINCT o.order_id) AS orders,
       COUNT(DISTINCT cu.customer_unique_id) AS customers,
       SUM(o.order_total_value) AS gmv_payments,
       AVG(o.order_total_value) AS aov,
       AVG(CASE WHEN o.is_delivered AND o.delivery_time_days IS NOT NULL THEN o.delivery_time_days END) AS avg_delivery_days
FROM analytics.fact_orders o
JOIN staging.customers cu ON o.customer_id = cu.customer_id
GROUP BY cu.customer_state
ORDER BY gmv_payments DESC
""").fetchdf()
geo.head(15)

,state,orders,customers,gmv_payments,aov,avg_delivery_days
0,SP,41746,40302,5998226.96,143.683873,8.700531
1,RJ,12852,12384,2144379.69,166.851828,15.237675
2,MG,11635,11259,1872257.26,160.915966,11.946543
3,RS,5466,5277,890898.54,162.989122,15.248503
4,PR,5045,4882,811156.38,160.784218,11.938046
5,SC,3637,3534,623086.43,171.318787,14.907527
6,BA,3380,3277,616645.82,182.439592,19.278563
7,DF,2140,2075,355141.08,165.953776,12.899038
8,GO,2020,1952,350092.31,173.313025,15.536025
9,ES,2033,1964,325967.55,160.338195,15.723810


In [4]:
# Reconciliation: all orders mapped
recon = conn.execute("""
SELECT (SELECT COUNT(*) FROM analytics.fact_orders) AS fact_orders,
       (SELECT COUNT(*) FROM analytics.fact_orders o JOIN staging.customers cu ON o.customer_id=cu.customer_id) AS mapped_orders,
       (SELECT SUM(gmv_payments) FROM (
          SELECT SUM(o.order_total_value) gmv_payments FROM analytics.fact_orders o JOIN staging.customers cu ON o.customer_id=cu.customer_id GROUP BY cu.customer_state)) AS gmv_sum
""").fetchdf()
recon

,fact_orders,mapped_orders,gmv_sum
0,99441,99441,16008872.12


In [5]:
conn.close()
